In [45]:
import funciones as fn
from pathlib import Path
from sklearn.model_selection import train_test_split
from scipy.stats import kstest, norm
import numpy as np

In [46]:
df_etl = fn.cargar_datos_limpios()
df_etl

,n_tokens_title,n_tokens_content,n_unique_tokens,n_non_stop_words,n_non_stop_unique_tokens,num_hrefs,num_self_hrefs,num_imgs,num_videos,average_token_length,...,min_positive_polarity,max_positive_polarity,avg_negative_polarity,min_negative_polarity,max_negative_polarity,title_subjectivity,title_sentiment_polarity,abs_title_subjectivity,abs_title_sentiment_polarity,shares
0,12.0,219.0,0.663594,1.0,0.815385,4.0,2.0,1.0,0.0,4.680365,...,0.100000,0.70,-0.350000,-0.600,-0.200000,0.500000,-0.187500,0.000000,0.187500,593
1,9.0,255.0,0.604743,1.0,0.791946,3.0,1.0,1.0,0.0,4.913725,...,0.033333,0.70,-0.118750,-0.125,-0.100000,0.000000,0.000000,0.500000,0.000000,711
2,9.0,211.0,0.575130,1.0,0.663866,3.0,1.0,1.0,0.0,4.393365,...,0.100000,1.00,-0.466667,-0.800,-0.133333,0.000000,0.000000,0.500000,0.000000,1500
3,9.0,531.0,0.503788,1.0,0.665635,9.0,0.0,1.0,0.0,4.404896,...,0.136364,0.80,-0.369697,-0.600,-0.166667,0.000000,0.000000,0.500000,0.000000,1200
4,13.0,1072.0,0.415646,1.0,0.540890,19.0,19.0,20.0,0.0,4.682836,...,0.033333,1.00,-0.220192,-0.500,-0.050000,0.454545,0.136364,0.045455,0.136364,505
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36418,11.0,346.0,0.529052,1.0,0.684783,9.0,7.0,1.0,1.0,4.523121,...,0.100000,0.75,-0.260000,-0.500,-0.125000,0.100000,0.000000,0.400000,0.000000,1800
36419,12.0,328.0,0.696296,1.0,0.885057,9.0,7.0,3.0,48.0,4.405488,...,0.136364,0.70,-0.211111,-0.400,-0.100000,0.300000,1.000000,0.200000,1.000000,1900
36420,10.0,442.0,0.516355,1.0,0.644128,24.0,1.0,12.0,1.0,5.076923,...,0.136364,0.50,-0.356439,-0.800,-0.166667,0.454545,0.136364,0.045455,0.136364,1900
36421,6.0,682.0,0.539493,1.0,0.692661,10.0,1.0,1.0,0.0,4.975073,...,0.062500,0.50,-0.205246,-0.500,-0.012500,0.000000,0.000000,0.500000,0.000000,1100


In [47]:
columnas_a_transformar = ['n_tokens_content', 'n_unique_tokens', 'num_hrefs', 'num_self_hrefs', 'num_imgs', 'num_videos', 'num_keywords', 'shares']
df_log = df_etl.copy()
# Aplica transformación logarítmica (usa log1p para evitar log(0))
for col in columnas_a_transformar:
    df_log[col + '_log'] = np.log1p(df_log[col])  # Crea nuevas columnas transformadas

for col in columnas_a_transformar:
    df_log.drop(columns=[col], inplace=True)
    df_log.rename(columns={col + '_log': col}, inplace=True)

In [48]:
df_log

,n_tokens_title,n_non_stop_words,n_non_stop_unique_tokens,average_token_length,data_channel_is_lifestyle,data_channel_is_entertainment,data_channel_is_bus,data_channel_is_socmed,data_channel_is_tech,data_channel_is_world,...,abs_title_subjectivity,abs_title_sentiment_polarity,n_tokens_content,n_unique_tokens,num_hrefs,num_self_hrefs,num_imgs,num_videos,num_keywords,shares
0,12.0,1.0,0.815385,4.680365,0.0,1.0,0.0,0.0,0.0,0.0,...,0.000000,0.187500,5.393628,0.508981,1.609438,1.098612,0.693147,0.000000,1.791759,6.386879
1,9.0,1.0,0.791946,4.913725,0.0,0.0,1.0,0.0,0.0,0.0,...,0.500000,0.000000,5.545177,0.472964,1.386294,0.693147,0.693147,0.000000,1.609438,6.568078
2,9.0,1.0,0.663866,4.393365,0.0,0.0,1.0,0.0,0.0,0.0,...,0.500000,0.000000,5.356586,0.454338,1.386294,0.693147,0.693147,0.000000,1.945910,7.313887
3,9.0,1.0,0.665635,4.404896,0.0,1.0,0.0,0.0,0.0,0.0,...,0.500000,0.000000,6.276643,0.407987,2.302585,0.000000,0.693147,0.000000,2.079442,7.090910
4,13.0,1.0,0.540890,4.682836,0.0,0.0,0.0,0.0,1.0,0.0,...,0.045455,0.136364,6.978214,0.347586,2.995732,2.995732,3.044522,0.000000,2.079442,6.226537
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36418,11.0,1.0,0.684783,4.523121,0.0,0.0,0.0,0.0,1.0,0.0,...,0.400000,0.000000,5.849325,0.424648,2.302585,2.079442,0.693147,0.693147,2.197225,7.496097
36419,12.0,1.0,0.885057,4.405488,0.0,0.0,0.0,1.0,0.0,0.0,...,0.200000,1.000000,5.796058,0.528447,2.302585,2.079442,1.386294,3.891820,2.079442,7.550135
36420,10.0,1.0,0.644128,5.076923,0.0,0.0,0.0,0.0,0.0,0.0,...,0.045455,0.136364,6.093570,0.416310,3.218876,0.693147,2.564949,0.693147,2.197225,7.550135
36421,6.0,1.0,0.692661,4.975073,0.0,0.0,0.0,0.0,0.0,1.0,...,0.500000,0.000000,6.526495,0.431453,2.397895,0.693147,0.693147,0.000000,1.791759,7.003974


In [49]:
random_state = 42

out_dir = Path('dataset')
out_dir.mkdir(parents=True, exist_ok=True)

# Separacion de 80% train, 20% temporal
train_df, val_df = train_test_split(
    df_log,
    test_size=0.2,
    random_state=random_state,
    shuffle=True
)

train_df.to_csv(out_dir / 'train.csv', index=False)
val_df.to_csv(out_dir / 'val.csv', index=False)